In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

In [ ]:

# ------------------------------
# Parameters
# ------------------------------
N = 20                     # Lattice size (N x N)
k_on = 1.0                 # Binding rate constant (per M/s)
k_off = 0.1                # Unbinding rate constant (per s)
c = 1.0                    # Ligand concentration (M)
alpha = 0.5                # Cooperative coupling factor for binding
beta = 0.0                 # (Optional) for unbinding cooperativity (at beta = 0, cooperativity doesnt matter)
t_max = 50.0               # Total simulation time
key = jax.random.PRNGKey(0)


In [ ]:

# ------------------------------
# Neighborhood Count
# ------------------------------
def count_bound_neighbors(lattice):
    """Count number of bound neighbors for each site."""
    up = jnp.roll(lattice, shift=1, axis=0)
    down = jnp.roll(lattice, shift=-1, axis=0)
    left = jnp.roll(lattice, shift=1, axis=1)
    right = jnp.roll(lattice, shift=-1, axis=1)
    return up + down + left + right


Description of terms:
- k_on = reaction rate for binding
- c = concentration (of the ligand)
- 1 + alpha * neighbors - cooperativity at that particular site
- is_unbound = identifies the bound state on the lattice

- k_off = reaction rate for unbinding
- 1 - beta * neighbors - cooperativity for unbinding (currently just 1)
- is_bound = identifies the unbound state on the lattice

In [ ]:

# ------------------------------
# Gillespie-like KMC Step
# ------------------------------
def kmc_step(lattice, t, key):

    # grid of size = lattice that tells us how many neighbors are bound at that site 
    neighbors = count_bound_neighbors(lattice) 

    # add boolean mask for binding affinity
    is_unbound = (lattice == 0)
    is_bound = (lattice == 1)

    # k_on = rate, c = concentration, 1 + alpha * neighbors = cooperativity
    rate_on = k_on * c * (1 + alpha * neighbors) * is_unbound
    rate_off = k_off * (1 - beta * neighbors) * is_bound
    rate_off = jnp.clip(rate_off, a_min=0.0)  # avoid negative rates

    # Combine into a single rate matrix (how likely is each position to flip it's state?)
    rates = rate_on + rate_off
    total_rate = rates.sum()

    # If total_rate is zero, stop simulation (if no more simulations can happen, be done)
    def no_op():
        return lattice, t + 1e6, key

    def do_update():
        # Pick a random time increment
        key1, key2, key3 = jax.random.split(key, 3)
        r = jax.random.uniform(key1)
        delta_t = -jnp.log(r) / total_rate

        # Flatten rates and sample a site
        flat_rates = rates.ravel()
        idx = jax.random.choice(key2, flat_rates.size, p=flat_rates / total_rate)
        i, j = jnp.unravel_index(idx, lattice.shape)

        # Flip state
        new_state = 1 - lattice[i, j]
        new_lattice = lattice.at[i, j].set(new_state)
        return new_lattice, t + delta_t, key3

    return jax.lax.cond(total_rate > 0, do_update, no_op)


In [ ]:
# ------------------------------
# Run Simulation
# ------------------------------
def run_kmc_lattice(t_max, key):
    lattice = jnp.zeros((N, N), dtype=int)
    t = 0.0
    snapshots = [lattice]
    times = [t]

    for _ in range(50):  # max steps
        print(_)
        lattice, t, key = kmc_step(lattice, t, key)
        if t > t_max:
            break
        snapshots.append(lattice)
        times.append(t)

    return jnp.array(times), jnp.stack(snapshots)


In [ ]:

# ------------------------------
# Run and Plot
# ------------------------------
times, snapshots = run_kmc_lattice(t_max, key)


In [ ]:

# Animate snapshots over time
import matplotlib.animation as animation

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(snapshots[0], cmap='Blues', vmin=0, vmax=1)
ax.set_title('Lattice Binding State Over Time')
ax.axis('off')

def update(frame):
    im.set_data(snapshots[frame])
    ax.set_title(f"Time: {times[frame]:.2f} s")
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(times), interval=5, blit=True)
plt.show()
